# Treinamento A2C — Super Mario Bros
## Notebook isolado por algoritmo (Fase 1)

Treina **A2C** em todas as fases × seeds. Output vai para diretório
compartilhado (`mario_drl_results/`) — Drive em Colab, local fora.

**Requisito:** Python 3.10 ou 3.11. No Colab Pro, garanta que sua runtime
não esteja em Python 3.12 (causa `OverflowError` em `nes-py`).

Após terminar todos os 3 algoritmos, abra `04_analysis.ipynb`.

## 1. Instalação
Rode UMA vez, depois RESTART.

In [ ]:
# ===================================================================
# Instalação — Colab (Python 3.10 ou 3.11) ou Linux local
# ===================================================================
# IMPORTANTE: este notebook requer Python 3.10 ou 3.11.
#   - nes-py 8.2.1 quebra em Python 3.12 (OverflowError em uint8)
#   - No Colab, garanta que está em Python 3.11 antes de rodar.
#
# Fluxo:
#   1) Rode esta célula UMA VEZ.
#   2) RESTART o kernel/sessão.
#   3) Rode a partir da célula 2.
# ===================================================================

import sys

PY = sys.version_info
print(f"Python: {PY.major}.{PY.minor}.{PY.micro}")
print(f"Executable: {sys.executable}")

if PY >= (3, 12):
    raise RuntimeError(
        f"\n⚠ Python {PY.major}.{PY.minor} não é suportado.\n"
        "   nes-py 8.2.1 quebra em Python 3.12+ (OverflowError em uint8).\n"
        "   No Colab: Runtime → Change runtime type → escolha Python 3.10 ou 3.11.\n"
        "   Localmente: use venv com Python 3.10/3.11."
    )

print(f"\n✓ Python {PY.major}.{PY.minor} é compatível. Instalando dependências (~3 min)...\n")

# Instalações progressivas — se algo falhar, fica claro qual passo foi
print(">> [1/5] numpy<2.0 (nes-py 8.2.1 quebra com NumPy 2.x)")
!pip install --quiet "numpy<2.0"

print(">> [2/5] RL stack (gym, gymnasium, shimmy, SB3)")
!pip install --quiet "gym==0.26.2" "gymnasium==0.29.1" "shimmy==1.3.0" "stable-baselines3==2.3.2"

print(">> [3/5] Super Mario Bros env (nes-py compila Cython, ~60s)")
!pip install --quiet "gym-super-mario-bros==7.4.0" "nes-py==8.2.1"

print(">> [4/5] Visualização (imageio, opencv)")
!pip install --quiet "imageio>=2.30" "opencv-python-headless>=4.8"

print(">> [5/5] PyTorch + utilitários (no-op se já tiver no Colab)")
!pip install --quiet "torch>=2.0,<2.5" "pandas" "scipy" "matplotlib" "seaborn" "tqdm" "tensorboard"

print("\n" + "="*60)
print("✓ Instalação concluída.")
print()
print("⚠  AGORA RESTART A SESSÃO antes de continuar:")
print("    Colab: Runtime → Restart session  (Ctrl+M .)")
print("    Local: Restart Kernel")
print("="*60)

## 2. Imports

In [ ]:
import os, json, time, random, pickle, warnings
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# RL stack
import gymnasium as gym
import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
from nes_py.wrappers import JoypadSpace

# Stable-Baselines3
from stable_baselines3 import DQN, PPO, A2C
from stable_baselines3.common.atari_wrappers import MaxAndSkipEnv, WarpFrame
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import (
    DummyVecEnv, SubprocVecEnv, VecFrameStack, VecMonitor
)
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, CallbackList

# Stats
from scipy.stats import spearmanr, mannwhitneyu

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
sns.set_theme(style="whitegrid", context="paper")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python : {os.sys.version_info.major}.{os.sys.version_info.minor}")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Storage (Colab Drive ou local)

In [ ]:
# ----------------------------------------------------------------------
# Storage: auto-detecta Colab → monta Drive; senão usa pasta local.
# Os 4 notebooks compartilham o MESMO diretório de resultados.
# ----------------------------------------------------------------------
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive", force_remount=False)
    ROOT_DIR = Path("/content/drive/MyDrive/mario_drl_results")
    print(f"Colab detectado — usando Drive: {ROOT_DIR}")
except ImportError:
    IN_COLAB = False
    ROOT_DIR = Path("./mario_drl_results").resolve()
    print(f"Local — usando: {ROOT_DIR}")

MODELS_DIR  = ROOT_DIR / "models"
LOGS_DIR    = ROOT_DIR / "logs"
TB_DIR      = ROOT_DIR / "tensorboard"
METRICS_DIR = ROOT_DIR / "metrics"
PLOTS_DIR   = ROOT_DIR / "plots"
for d in (MODELS_DIR, LOGS_DIR, TB_DIR, METRICS_DIR, PLOTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 4. Configuração global

In [ ]:
# ----------------------------------------------------------------------
# Parâmetros globais do experimento
# ----------------------------------------------------------------------
SMOKE_TEST = True   # True = teste rápido (~2 min); False = experimento real

STAGES = ["1-1", "1-2", "4-1", "8-1"]
SEEDS  = [42, 123, 2024]
ALGOS  = ["A2C"]

if SMOKE_TEST:
    TOTAL_TIMESTEPS = 10_000
    EVAL_FREQ       = 2_000
    N_EVAL_EPISODES = 2
    STAGES_TO_RUN   = ["1-1"]
    SEEDS_TO_RUN    = [42]
else:
    TOTAL_TIMESTEPS = 500_000
    EVAL_FREQ       = 10_000
    N_EVAL_EPISODES = 5
    STAGES_TO_RUN   = STAGES
    SEEDS_TO_RUN    = SEEDS

FRAME_STACK = 4
print(f"Modo: {'SMOKE_TEST' if SMOKE_TEST else 'COMPLETO'}  |  "
      f"Treinamentos: {len(ALGOS) * len(STAGES_TO_RUN) * len(SEEDS_TO_RUN)}  |  "
      f"Timesteps/treino: {TOTAL_TIMESTEPS:,}")

## 5. Hiperparâmetros — A2C

In [ ]:
HPARAMS = {
    "A2C": dict(
        learning_rate     = 7e-4,
        n_steps           = 5,
        gamma             = 0.99,
        gae_lambda        = 1.0,
        ent_coef          = 0.01,
        vf_coef           = 0.25,
        max_grad_norm     = 0.5,
        rms_prop_eps      = 1e-5,
        use_rms_prop      = True,
        n_envs            = 16,
    ),
}

## 6. Environment factory

In [ ]:
# Force shim V21 — JoypadSpace usa API antiga (gym pre-0.26)
import gym as _legacy_gym
from gym.wrappers import TimeLimit as _GymTimeLimit
from shimmy import GymV21CompatibilityV0 as _GymCompat
print(f"gym {_legacy_gym.__version__} → shimmy.{_GymCompat.__name__} (forçado V21)")


def _strip_time_limit(env):
    """
    Remove o TimeLimit que gym.make() envelopa automaticamente em 0.26+.
    Esse wrapper espera API nova (5-tuple step), mas SuperMarioBrosEnv
    retorna 4-tuple. Removendo-o, deixamos o env "cru" passar pro JoypadSpace.
    """
    while isinstance(env, _GymTimeLimit):
        env = env.env
    return env


class CompatJoypadSpace(JoypadSpace):
    """JoypadSpace tolerante a (seed, options) no reset."""
    def reset(self, seed=None, options=None, **kwargs):
        return super().reset(**kwargs)


def make_mario_env(stage: str = "1-1", seed: int = 0):
    env_id = f"SuperMarioBros-{stage}-v0"
    env = gym_super_mario_bros.make(env_id)
    env = _strip_time_limit(env)            # remove TimeLimit antes do JoypadSpace
    env = CompatJoypadSpace(env, SIMPLE_MOVEMENT)
    env = _GymCompat(env=env)               # converte API antiga → gymnasium nova
    env = MaxAndSkipEnv(env, skip=4)
    env = WarpFrame(env, width=84, height=84)
    env = Monitor(env)
    env.action_space.seed(seed)
    return env


def make_vec_env_mario(stage: str, n_envs: int, seed: int, use_subproc: bool = True):
    def make_one(rank):
        def _init():
            return make_mario_env(stage=stage, seed=seed + rank)
        return _init
    env_fns = [make_one(i) for i in range(n_envs)]
    if n_envs > 1 and use_subproc:
        vec_env = SubprocVecEnv(env_fns, start_method="fork")
    else:
        vec_env = DummyVecEnv(env_fns)
    vec_env = VecFrameStack(vec_env, n_stack=FRAME_STACK)
    vec_env = VecMonitor(vec_env)
    return vec_env


def set_global_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## 7. Callback de avaliação

In [ ]:
class MarioEvalCallback(BaseCallback):
    """Avalia periodicamente e registra métricas Grupo I + II em CSV."""
    def __init__(self, eval_stage, eval_freq, n_eval_episodes, log_path, seed=0, verbose=1):
        super().__init__(verbose)
        self.eval_stage = eval_stage
        self.eval_freq = eval_freq
        self.n_eval_episodes = n_eval_episodes
        self.log_path = Path(log_path)
        self.seed = seed
        self.eval_env = None
        self._rows = []

    def _on_training_start(self):
        self.eval_env = make_vec_env_mario(
            stage=self.eval_stage, n_envs=1, seed=self.seed + 9999, use_subproc=False
        )

    def _run_eval(self):
        results = []
        for ep in range(self.n_eval_episodes):
            obs = self.eval_env.reset()
            done = [False]
            ep_reward, max_x, deaths, frames = 0.0, 0, 0, 0
            flag_get = False
            time_left = None
            prev_life = None
            while not done[0]:
                action, _ = self.model.predict(obs, deterministic=True)
                obs, reward, done, info = self.eval_env.step(action)
                info0 = info[0]
                ep_reward += float(reward[0])
                frames += 1
                x = int(info0.get("x_pos", 0))
                if x > max_x: max_x = x
                life = info0.get("life", None)
                if prev_life is not None and life is not None and life < prev_life:
                    deaths += 1
                prev_life = life
                if info0.get("flag_get", False): flag_get = True
                time_left = info0.get("time", time_left)
            results.append(dict(
                episode=ep, reward=ep_reward, max_x_pos=max_x,
                flag_get=int(flag_get), deaths=deaths, frames=frames, time_left=time_left
            ))
        return results

    def _on_step(self):
        if self.num_timesteps % self.eval_freq != 0:
            return True
        eval_results = self._run_eval()
        for r in eval_results:
            self._rows.append({"timestep": int(self.num_timesteps), **r})
        rewards = [r["reward"] for r in eval_results]
        flags   = [r["flag_get"] for r in eval_results]
        if self.verbose >= 1:
            print(f"  [eval @ {self.num_timesteps:>7,}] "
                  f"reward={np.mean(rewards):+8.2f}±{np.std(rewards):5.2f}  "
                  f"completion={np.mean(flags):.0%}  "
                  f"max_x={np.mean([r['max_x_pos'] for r in eval_results]):.0f}")
        pd.DataFrame(self._rows).to_csv(self.log_path, index=False)
        return True

    def _on_training_end(self):
        if self.eval_env is not None:
            self.eval_env.close()

## 8. Função de treinamento

In [ ]:
import re

def experiment_id(algo: str, stage: str, seed: int) -> str:
    return f"{algo}_stage{stage}_seed{seed}"


def _find_latest_checkpoint(exp_id: str):
    """
    Procura o checkpoint mais recente para um exp_id.
    Retorna (path, timesteps_done) ou (None, 0) se não houver.

    Checkpoints são salvos pelo CheckpointCallback como:
      MODELS_DIR/checkpoints/{exp_id}_{N}_steps.zip
    """
    ckpt_dir = MODELS_DIR / "checkpoints"
    if not ckpt_dir.exists():
        return None, 0

    pattern = re.compile(rf"^{re.escape(exp_id)}_(\d+)_steps\.zip$")
    candidates = []
    for p in ckpt_dir.glob(f"{exp_id}_*_steps.zip"):
        m = pattern.match(p.name)
        if m:
            candidates.append((int(m.group(1)), p))

    if not candidates:
        return None, 0
    candidates.sort()
    steps, path = candidates[-1]   # maior N (mais recente)
    return path, steps


def train_one(algo: str, stage: str, seed: int, overwrite: bool = False,
              save_checkpoints: bool = True, n_checkpoints: int = 5):
    """
    Treina UM agente (algo, stage, seed). Idempotente E retomável:

    - Se já existe modelo final → pula (a menos que overwrite=True)
    - Se NÃO existe modelo final mas existe um checkpoint parcial → RETOMA
      carregando do checkpoint e treinando só os timesteps restantes
    - Caso contrário → começa do zero
    """
    exp_id   = experiment_id(algo, stage, seed)
    log_csv  = LOGS_DIR / f"{exp_id}.csv"
    model_pt = MODELS_DIR / f"{exp_id}.zip"
    tb_path  = TB_DIR / exp_id
    ckpt_dir = MODELS_DIR / "checkpoints"

    # Treino completo já feito?
    if log_csv.exists() and model_pt.exists() and not overwrite:
        print(f"  → {exp_id} já completo, pulando.")
        return log_csv

    set_global_seed(seed)
    hp = HPARAMS[algo].copy()
    n_envs = hp.pop("n_envs")

    train_env = make_vec_env_mario(stage=stage, n_envs=n_envs, seed=seed,
                                   use_subproc=(n_envs > 1))

    # ---- LÓGICA DE RESUME -------------------------------------------------
    ckpt_path, ckpt_steps = _find_latest_checkpoint(exp_id)
    cls = {"DQN": DQN, "PPO": PPO, "A2C": A2C}[algo]

    if ckpt_path is not None and not overwrite and ckpt_steps < TOTAL_TIMESTEPS:
        print(f"  ↻ Retomando {exp_id} do checkpoint: {ckpt_path.name} ({ckpt_steps:,} steps)")
        model = cls.load(str(ckpt_path), env=train_env, device=device,
                         tensorboard_log=str(tb_path))
        remaining = TOTAL_TIMESTEPS - ckpt_steps
        reset_num_timesteps = False  # mantém contagem do TensorBoard
    else:
        common_kwargs = dict(policy="CnnPolicy", env=train_env, verbose=0,
                             seed=seed, device=device, tensorboard_log=str(tb_path))
        model = cls(**common_kwargs, **hp)
        remaining = TOTAL_TIMESTEPS
        reset_num_timesteps = True
    # -----------------------------------------------------------------------

    eval_cb = MarioEvalCallback(
        eval_stage=stage, eval_freq=max(EVAL_FREQ // n_envs, 1),
        n_eval_episodes=N_EVAL_EPISODES, log_path=log_csv, seed=seed, verbose=1,
    )
    callbacks = [eval_cb]

    if save_checkpoints and not SMOKE_TEST:
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        ckpt_save_freq = max(TOTAL_TIMESTEPS // n_envs // n_checkpoints, 1)
        callbacks.append(CheckpointCallback(
            save_freq=ckpt_save_freq, save_path=str(ckpt_dir),
            name_prefix=exp_id, save_replay_buffer=False, save_vecnormalize=False,
        ))

    cb = CallbackList(callbacks) if len(callbacks) > 1 else callbacks[0]

    t0 = time.time()
    print(f"\n► [{exp_id}] treinando ({remaining:,} timesteps restantes, n_envs={n_envs})")
    try:
        model.learn(total_timesteps=remaining, callback=cb,
                    tb_log_name=exp_id, progress_bar=True,
                    reset_num_timesteps=reset_num_timesteps)
        model.save(model_pt)
        print(f"✓ [{exp_id}] concluído em {(time.time()-t0)/60:.1f} min")
    finally:
        train_env.close()
    return log_csv

## 9. Loop de treinamento — A2C

Roteiro:
1. Com `SMOKE_TEST = True` (célula 4) → valida pipeline em ~2 min
2. Mude para `SMOKE_TEST = False`, reinicie, descomente a chamada abaixo

**Idempotente:** se cair, é só rerodar — pula o que já completou.

In [ ]:
def run_a2c_experiments():
    """Treina A2C em todas as fases × seeds. Idempotente."""
    total = len(STAGES_TO_RUN) * len(SEEDS_TO_RUN)
    done = 0
    for stage in STAGES_TO_RUN:
        for seed in SEEDS_TO_RUN:
            done += 1
            print(f"\n{'='*70}\n[{done}/{total}] A2C | stage {stage} | seed {seed}\n{'='*70}")
            try:
                train_one(algo="A2C", stage=stage, seed=seed)
            except Exception as e:
                print(f"✗ FALHA [A2C/{stage}/{seed}]: {e}")
    print("\n✓ Loop encerrado.")

# Descomente para rodar:
# run_a2c_experiments()
print("Pronto. Descomente a chamada acima para iniciar os treinamentos.")

## 10. Verificação visual — renderiza A2C jogando

In [ ]:
import imageio.v2 as imageio
from matplotlib import animation as _mpl_animation
from IPython.display import HTML, display


def animate_frames_inline(frames, fps: int = 15, figsize=(6, 5)):
    """Anima frames inline via matplotlib.to_jshtml() — funciona em qualquer cliente."""
    if not frames:
        return HTML("<i>Nenhum frame.</i>")
    fig, ax = plt.subplots(figsize=figsize); ax.axis("off")
    im = ax.imshow(frames[0])
    def update(i):
        im.set_array(frames[i])
        return [im]
    ani = _mpl_animation.FuncAnimation(fig, update, frames=len(frames),
                                       interval=1000.0/fps, blit=True)
    html = ani.to_jshtml(default_mode="loop")
    plt.close(fig)
    return HTML(html)


def render_agent_episode(model, stage="1-1", max_steps=2000, fps=15, seed=999):
    env_id = f"SuperMarioBros-{stage}-v0"
    base = gym_super_mario_bros.make(env_id, render_mode="rgb_array")
    base = CompatJoypadSpace(base, SIMPLE_MOVEMENT)
    base = _GymCompat(env=base)
    base = MaxAndSkipEnv(base, skip=4)
    base = WarpFrame(base, width=84, height=84)
    base = Monitor(base)
    vec_env = DummyVecEnv([lambda: base])
    vec_env = VecFrameStack(vec_env, n_stack=FRAME_STACK)
    render_env = vec_env.venv.envs[0]
    frames, metrics = [], dict(reward=0.0, max_x=0, deaths=0, flag_get=False, steps=0)
    prev_life = None
    obs = vec_env.reset()
    for _ in range(max_steps):
        f = render_env.render()
        if f is not None: frames.append(f.copy())
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = vec_env.step(action)
        info0 = info[0]
        metrics["reward"] += float(reward[0]); metrics["steps"] += 1
        x = int(info0.get("x_pos", 0))
        if x > metrics["max_x"]: metrics["max_x"] = x
        life = info0.get("life", None)
        if prev_life is not None and life is not None and life < prev_life:
            metrics["deaths"] += 1
        prev_life = life
        if info0.get("flag_get", False): metrics["flag_get"] = True
        if done[0]:
            f = render_env.render()
            if f is not None: frames.append(f.copy())
            break
    vec_env.close()
    out_path = PLOTS_DIR / f"agent_A2C_{stage}.gif"
    imageio.mimsave(out_path, frames, duration=int(round(1000.0/fps)), loop=0)
    print(f"✓ {len(frames)} frames → {out_path}")
    print(f"   reward={metrics['reward']:+.1f}  max_x={metrics['max_x']}  "
          f"flag={metrics['flag_get']}  deaths={metrics['deaths']}")
    return out_path, metrics, frames


# Demonstração: carrega modelo A2C treinado e mostra animação
model_path = MODELS_DIR / f"A2C_stage1-1_seed42.zip"
if model_path.exists():
    model = A2C.load(model_path, device=device)
    _, _, frames = render_agent_episode(model, stage="1-1", max_steps=500, seed=999)
    display(animate_frames_inline(frames, fps=15))
else:
    print(f"Modelo não encontrado: {model_path.name}. Rode o treinamento antes.")

## 11. Próximo passo

Quando todos os algoritmos terminarem, abra **`04_analysis.ipynb`** para:
- Carregar todos os logs salvos em `LOGS_DIR`
- Computar métricas dos Grupos I e II
- Gerar gráficos comparativos
- Comparações visuais lado-a-lado e evolução temporal